# Feature map 到 attention

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/15-attention-bridge/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍與後續實驗〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [ ]:
"""An actual QK-softmax-V calculation with the spatial round trip."""
import math
import torch
from torch import nn
from torch.nn import functional as F


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    feature = torch.tensor([[[[1., 0.], [1., 0.]], [[0., 1.], [1., 0.]]]])  # [1,2,2,2]
    tokens = feature.flatten(2).transpose(1, 2)  # [1,4,2] in row-major spatial order
    qkv = nn.Linear(2, 6, bias=False)
    with torch.no_grad():
        qkv.weight.copy_(torch.eye(2).repeat(3, 1))
    q, k, v = qkv(tokens).chunk(3, dim=-1)
    attention = (q @ k.transpose(-2, -1) / math.sqrt(2)).softmax(-1)
    output_tokens = attention @ v
    output = output_tokens.transpose(1, 2).reshape_as(feature)
    assert tokens[0].tolist() == [[1., 0.], [0., 1.], [1., 1.], [0., 0.]]
    assert torch.allclose(attention.sum(-1), torch.ones(1, 4))
    row = torch.tensor([1., 0., 1., 0.]).div(math.sqrt(2)).softmax(0)
    assert torch.allclose(attention[0, 0], row)
    assert torch.allclose(output_tokens[0, 0], torch.tensor([row[0] + row[2], row[1] + row[2]]))
    print('tokens:', tokens.tolist())
    # round in float64 so 0.3349 prints as 0.3349, not as float32's nearest value 0.3348999917...
    print('first attention row:', attention[0, 0].detach().double().round(decimals=4).tolist())
    print('first weighted value:', output_tokens[0, 0].detach().double().round(decimals=4).tolist())
    print('shape feature -> tokens -> affinity -> feature:', tuple(feature.shape), tuple(tokens.shape), tuple(attention.shape), tuple(output.shape))
    # Exercise runs before SGD, while the shared projection is still identity.
    # Keep all original-input assertions intact.
    with torch.no_grad():
        exercise_tokens = tokens.detach().clone()
        exercise_tokens[0, 3] = torch.tensor([2., 0.])
        eq, ek, ev = qkv(exercise_tokens).chunk(3, dim=-1)
        exercise_weights = (eq @ ek.transpose(-2, -1) / math.sqrt(2)).softmax(-1)
        exercise_output = exercise_weights @ ev
        expected_row = torch.tensor([1., 0., 1., 2.]).div(math.sqrt(2)).softmax(0)
        assert torch.allclose(exercise_weights[0, 0], expected_row)
        assert torch.allclose(exercise_output[0, 0], torch.tensor([1.3395, .3302]), atol=1e-4)
        print('exercise fourth-token [2,0], first weight row:', exercise_weights[0, 0].double().round(decimals=4).tolist())
        print('exercise first output:', exercise_output[0, 0].double().round(decimals=4).tolist())
    optimizer = torch.optim.SGD(qkv.parameters(), lr=.1)
    optimizer.zero_grad()
    # Halve channel 1: with target=feature the example is symmetric under a channel swap,
    # so from the identity start Q and K would get identical gradients.
    target = feature * torch.tensor([1., .5]).view(1, 2, 1, 1)
    loss = F.mse_loss(output, target)
    loss.backward()
    grad_q, grad_k, grad_v = qkv.weight.grad.chunk(3, dim=0)
    assert all(part.abs().sum() > 0 for part in (grad_q, grad_k, grad_v))
    # allclose, not equal: rounding alone can make mathematically equal gradients differ.
    assert not torch.allclose(grad_q, grad_k)
    assert not torch.allclose(grad_q, grad_v)
    assert not torch.allclose(grad_k, grad_v)
    old = qkv.weight.detach().clone()
    optimizer.step()
    assert not torch.equal(old, qkv.weight)
    print(f'backward/step through Q,K,V verified; reconstruction loss={loss.item():.4f}')


if __name__ == '__main__':
    main()
